# Spatiotemporal shotgun screen — XGBoost

A reduced-compute **first-stage screen** of nonlinear spatiotemporal ideas that can be added to your current XGBoost **47-term binary BoW** model.

### Stage A: append or replace features

**10 configurations:** existing baseline, age-first comps, property-type quota, both, lagged market indices, multiscale comps, approximate time-adjusted comp, spatial-temporal Fourier/RBF features, nonlinear spline/tensor basis, and the combined feature package. All are fitted on the same sampled historical sale cohort and scored on the same held-out month.

### Stage B: fit residual corrections on earlier out-of-time predictions

**8 configurations:** kNN, local Gaussian kernel, RBF-ridge, additive spline GAM approximation, tensor-spline GAM approximation, ExtraTrees, histogram boosting, and a small Gaussian process. The residual-model targets come from earlier months predicted by base XGBoost models trained only on older sale months. Stage B corrects the **same baseline predictions** made in Stage A.

This is not a full geostatistical GAM, STARIMA, NNGP, DeepKriging or graph-neural implementation. It screens inexpensive representatives from relevant families before we invest in heavier standalone models.

**Crucial limitations:** the sample/trees are intentionally smaller than your full six-month XGBoost benchmark; numbers are **not** expected to reproduce 7.82% MdAPE. The 500-ish homes in the selected October validation fold are reused for ranking variants. Validate only a frozen shortlist on a different historical fold and eventually the full-data confirmation/fixed-origin runs. No results will be written to `logs.md` automatically.

In [ ]:
import pandas as pd
from IPython.display import display
from spacetime_shotgun import run_shotgun, SCREEN_FEATURE_VARIANTS
from models.spacetime_screen_residuals import RESIDUAL_VARIANTS

VALIDATION_MONTH = "2025-10"  # must be earlier than December 2025
TRAIN_START = "2024-01-01"
TRAIN_CAP = 10000              # random sample of historical sales
VALIDATION_CAP = 650           # holdout sales, excluded from ALL training
CALIBRATION_CAP = 650          # per earlier calibration month
CALIBRATION_FOLDS = 2         # August and September 2025 by default
SCREEN_TREES = 220            # not the full tuned 700 trees
RESIDUAL_STRENGTH = 0.5
SEED = 42

FEATURES = list(SCREEN_FEATURE_VARIANTS)
RESIDUALS = list(RESIDUAL_VARIANTS)
print("Feature candidates:", FEATURES)
print("Residual candidates:", RESIDUALS)

In [ ]:
scores, predictions, failures = run_shotgun(
    validation_month=VALIDATION_MONTH,
    train_start=TRAIN_START,
    train_cap=TRAIN_CAP,
    validation_cap=VALIDATION_CAP,
    calibration_cap=CALIBRATION_CAP,
    calibration_folds=CALIBRATION_FOLDS,
    screen_trees=SCREEN_TREES,
    seed=SEED,
    features=FEATURES,
    residuals=RESIDUALS,
    residual_strength=RESIDUAL_STRENGTH,
)
display(scores)
if failures:
    print(f"{len(failures)} failed candidates; see outputs/spacetime_shotgun/failures.json.")
    display(pd.DataFrame(failures)[["arm", "variant", "error"]])

In [ ]:
# Feature and residual experiments must be compared separately to baseline.
display(scores.sort_values(["arm", "mdape"]))
display(
    scores.groupby("arm")[["mdape", "mape", "delta_mdape_pp", "delta_mape_pp"]]
    .agg(["min", "median"]).round(3)
)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
p = scores.sort_values("delta_mdape_pp")
ax.barh(p["arm"] + " / " + p["variant"], p["delta_mdape_pp"])
ax.axvline(0, linewidth=1, color="gray")
ax.set_xlabel("MdAPE change vs reduced-sample baseline (percentage points)")
ax.set_title("Development shotgun screen · lower is better")
fig.tight_layout()
plt.show()

In [ ]:
# Check whether proposed fixes help difficult property categories rather
# than only changing aggregate error. This is an optional diagnostic; the
# ranking itself is based on held-out prices from one month.
import data
sales = data.load("listings")
sold_month = pd.to_datetime(sales["date_sold"]).dt.to_period("M")
names = sales.loc[sold_month == pd.Period(VALIDATION_MONTH), [
    "ml_num", "house_type_name"
]]
annotated = predictions.merge(names, on="ml_num", how="left", validate="many_to_one")
subgroups = (
    annotated.groupby(["arm", "variant", "house_type_name"])
    .agg(n=("ml_num", "size"), mdape=("abs_pct_error", "median"))
    .query("n >= 10")
    .sort_values(["house_type_name", "mdape"])
)
display(subgroups.head(40))
print("Full result CSV: outputs/spacetime_shotgun/predictions.csv")

## Second-stage research after the screen

Once a small number of configurations survive on October 2025, switch `VALIDATION_MONTH` to another **earlier** month, keeping settings fixed. Check stability, errors by house type, and fixed-origin issues.

A later **standalone/residual model comparison** can then evaluate more expensive nonlinear methods (geographically weighted random forest, DeepKriging, a genuine spatiotemporal GAM/tensor smoother, scalable NNGP, spatiotemporal GNN, and neural autoregressive regional models). Those methods need their own honest, comparable training protocol; many also require aggregating transactions into regional monthly series. They have **not** been implemented as part of this quick screen.

### Installation and test

```powershell
git pull origin main
uv sync
uv run python -m pytest tests/test_spacetime_shotgun.py -q
uv run jupyter lab
```

You can also run the screen without Jupyter:

```powershell
uv run python spacetime_shotgun.py --validation-month 2025-10
```

To reduce initial compute further, set `TRAIN_CAP=5000`, `VALIDATION_CAP=250`, `SCREEN_TREES=100`, and choose a subset of `FEATURES` and `RESIDUALS`. Do not infer small score changes from these very small samples.